# Analysis of reversal region
## A. Ordog, Mar 2024
### Sept 3: need to update this for use with new map files!

In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
import numpy as np
import matplotlib.pyplot as plt
from astropy.wcs import WCS
import gc
from astropy.coordinates import SkyCoord, ICRS, Galactic
from astropy import units as u

In [ ]:
def read_in_PI(directory, file, return_hdr = False):
    
    hdu  = fits.open(directory+file)
    PI = hdu[0].data
    
    print(PI.shape)
    
    if return_hdr:
        hdr = hdu[0].header
    else:
        hdr = None
        
    return PI, hdr


def read_in_RM(directory, file, return_hdr = False):
    
    hdu  = fits.open(directory+file)
    
    if return_hdr:
        hdr = hdu[0].header
    else:
        hdr = None

    data = {'RM' : hdu[0].data,
            'R'  : hdu[2].data,
            'sig': hdu[4].data}

    data['RM'][hdu[0].data==0] = np.nan
    data['R'][hdu[0].data==0]  = np.nan
    data['sig'][hdu[0].data==0] = np.nan

    print(data['RM'].shape)
        
    return data, hdr

In [ ]:
def read_st_catalogs(directory, file):
    
    catalog = fits.open(directory+file)
    
    st_cat = {}
    lon = []
    lat = []
    RM  = []
    
    for i in range(0,catalog[1].data.shape[0]):
        lon.append(catalog[1].data[i][2])
        lat.append(catalog[1].data[i][3])
        RM.append( catalog[1].data[i][5])
    
    st_cat['lon'] = np.array(lon)
    st_cat['lat'] = np.array(lat)
    st_cat['RM'] = np.array(RM)
    
    return st_cat

In [ ]:
def make_map_plots(lon_range = [82,52],lat_range = [-6,8],vmax = 200):
    
    fs = 12
    crange = SkyCoord(lon_range, lat_range, frame=Galactic, unit=(u.deg, u.deg))

    fig = plt.figure(figsize=(20,10))
    plt.subplots_adjust(wspace=0, hspace=0)
    ax1 = fig.add_subplot(221, projection=WCS(hdr).celestial)
    ax2 = fig.add_subplot(222, projection=WCS(hdr).celestial)
    ax3 = fig.add_subplot(223, projection=WCS(hdr).celestial)
    ax4 = fig.add_subplot(224, projection=WCS(hdr).celestial)

    im1 = ax1.imshow(FD_G,vmin=-vmax,vmax=vmax,cmap='RdBu_r')
    im2 = ax2.imshow(RM_data_G['RM'],vmin=-vmax,vmax=vmax,cmap='RdBu_r')
    im3 = ax3.imshow(RM_data_C['RM'],vmin=-vmax,vmax=vmax,cmap='RdBu_r')
    im4 = ax4.imshow(RM_data_CG['RM'],vmin=-vmax,vmax=vmax,cmap='RdBu_r')

    axs = [ax1, ax2, ax3, ax4]
    ims = [im1, im2, im3, im4]

    ipt = []
    jpt = []
    for i in range(0,len(st_cat['lon'])):
        c = SkyCoord(st_cat['lon'][i],st_cat['lat'][i], frame=Galactic, unit="deg")
        pixels = WCS(hdr).world_to_pixel(c)
        ipt.append(int(np.round(pixels[0],0)))
        jpt.append(int(np.round(pixels[1],0)))

    for i in range(0,4):
        axs[i].set_ylabel(' ',fontsize=fs)
        axs[i].set_xlabel(' ',fontsize=fs)
        axs[i].set_xlim(WCS(hdr).world_to_pixel(crange)[0])
        axs[i].set_ylim(WCS(hdr).world_to_pixel(crange)[1])
        axs[i].tick_params(axis='both', which='major', labelsize=fs, zorder=30, length=5)
        axs[i].scatter(ipt,jpt,c=st_cat['RM'],vmin=-vmax,vmax=vmax,s=60,cmap='RdBu_r',edgecolors= "k")

        #cbar = fig.colorbar(ims[i],ax=axs[i],orientation='vertical',fraction=0.012,pad=0.02,aspect=20)
        #cbar.ax.tick_params(axis='y', which='both',labelsize=fs)
        #cbar.set_label('[rad]', fontsize=fs)

    return

In [ ]:
def calc_diffuse_beams(R1,R2,lon_pt,lat_pt):

    center_coord = SkyCoord(lon_pt, lat_pt, frame=Galactic, unit="deg")
    pixels = WCS(hdr).world_to_pixel(center_coord)
    y, x = np.indices((RM_CG.shape))
    coords = WCS(hdr).pixel_to_world(x, y)
    r_dist = center_coord.separation(coords).deg
    
    RM_beam_C = np.nanmean(RM_C[(r_dist <= R2) & (r_dist >= R1)])
    RM_beam_G = np.nanmean(RM_G[(r_dist <= R2) & (r_dist >= R1)])
    RM_beam_CG = np.nanmean(RM_CG[(r_dist <= R2) & (r_dist >= R1)])
    FD_beam_G = np.nanmean(FD_G[(r_dist <= R2) & (r_dist >= R1)])
    
    return RM_beam_C, RM_beam_G, RM_beam_CG, FD_beam_G

## Read in data:

In [ ]:
#st_cat = read_st_catalogs('/home/aordog/DATA/CGPS_catalog/', 'CGPS_RMTable.fits')
st_cat = read_st_catalogs('/srv/data/cgps/', 'CGPS_RMTable.fits')

#directory = '/home/aordog/DATA/cgps-gmims/'
directory = '/srv/data/cgps-gmims_2023/'
RM_data_C,  hdr = read_in_RM(directory,'RM_C_conv4_regrd.fits',return_hdr=False)
#RM_data_G,  hdr = read_in_RM(directory,'RM_G_conv4_regrd.fits',return_hdr=False)
RM_data_CG, hdr = read_in_RM(directory,'RM_CG_conv4_regrd.fits',return_hdr=True)

#FD_G = fits.open(directory+'phi_peak_regrd.fits')[0].data
#print(FD_G.shape)

directory = '/srv/data/cgps-gmims_2023/'
PI_C,  hdr = read_in_PI(directory,'PI_C_conv4_regrd_avg_PI.fits',return_hdr=False)
#PI_G,  hdr = read_in_PI(directory,'PI_G_regrd_avg_PI.fits',return_hdr=False)
PI_CG, hdr = read_in_PI(directory,'PI_CG_conv4_regrd_avg_PI.fits',return_hdr=True)


## Make maps with point sources RMs overplotted

In [ ]:
make_map_plots(lon_range = [82,52],lat_range = [-6,8],vmax = 200)

## Calculate diffuse emission RMs/FDs in annuli around point sources:

In [ ]:
########################################################
lon_max = 82
R2      = 20/60 # 20 arcmin radius (GMIMS-HBN beam)
R1      = 2/60  # exclude 2 arcmin radius around source
########################################################

lon_sub = st_cat['lon'][st_cat['lon'] <= lon_max]
lat_sub = st_cat['lat'][st_cat['lon'] <= lon_max]
RM_sub = st_cat['RM'][st_cat['lon'] <= lon_max]

beams = {'CG_RM':[],
         'C_RM':[],
         'G_RM':[],
         'G_FD':[]}

for i in range(0,len(lon_sub)):
    
    print(i,'of',len(lon_sub),lon_sub[i],lat_sub[i])
    RM_beam_C,RM_beam_G,RM_beam_CG,FD_beam_G = calc_diffuse_beams(R1,R2,lon_sub[i],lat_sub[i])
    
    beams['CG_RM'].append(RM_beam_CG)
    beams['C_RM'].append(RM_beam_C)
    beams['G_RM'].append(RM_beam_G)
    beams['G_FD'].append(FD_beam_G)
    

## Diffuse RMs/FDs versus point source RMs

In [ ]:
fix,axs = plt.subplots(2,2,figsize=(12,12))

axs[0,0].scatter(RM_sub,beams['G_FD'])
axs[0,1].scatter(RM_sub,beams['G_RM'])
axs[1,0].scatter(RM_sub,beams['C_RM'])
axs[1,1].scatter(RM_sub,beams['CG_RM'])

axs[0,0].set_ylabel(r'GMIMS peak FD (rad/m$^2$)')
axs[0,1].set_ylabel(r'GMIMS linear fit RM (rad/m$^2$)')
axs[1,0].set_ylabel(r'CGPS RM (rad/m$^2$)')
axs[1,1].set_ylabel(r'CGPS+GMIMS RM (rad/m$^2$)')

for i in range(0,2):
    for j in range(0,2):
        axs[j,i].set_xlim(-600,600)
        axs[j,i].set_ylim(-600,600)
        axs[j,i].set_xlabel(r'CGPS point source RM (rad/m$^2$)')

## Map the ratios of point source RMs to diffuse RMs/FDs

In [ ]:
lon_range = [82,52]
lat_range = [-6,8]
fs = 12
vmax = 20

crange = SkyCoord(lon_range, lat_range, frame=Galactic, unit=(u.deg, u.deg))

fig = plt.figure(figsize=(20,10))
plt.subplots_adjust(wspace=0, hspace=0)
ax1 = fig.add_subplot(221, projection=WCS(hdr).celestial)
ax2 = fig.add_subplot(222, projection=WCS(hdr).celestial)
ax3 = fig.add_subplot(223, projection=WCS(hdr).celestial)
ax4 = fig.add_subplot(224, projection=WCS(hdr).celestial)

axs = [ax1, ax2, ax3, ax4]
ims = [im1, im2, im3, im4]

ipt = []
jpt = []
for i in range(0,len(lon_sub)):
    #print(i)
    c = SkyCoord(lon_sub[i],lat_sub[i], frame=Galactic, unit="deg")
    pixels = WCS(hdr).world_to_pixel(c)
    ipt.append(int(np.round(pixels[0],0)))
    jpt.append(int(np.round(pixels[1],0)))

print(c)

ax1.scatter(ipt,jpt,c=RM_sub/beams['G_RM'],vmin=-vmax,vmax=vmax,s=60,cmap='RdBu_r',edgecolors= "k")
ax2.scatter(ipt,jpt,c=RM_sub/beams['G_FD'],vmin=-vmax,vmax=vmax,s=60,cmap='RdBu_r',edgecolors= "k")
ax3.scatter(ipt,jpt,c=RM_sub/beams['C_RM'],vmin=-vmax,vmax=vmax,s=60,cmap='RdBu_r',edgecolors= "k")
ax4.scatter(ipt,jpt,c=RM_sub/beams['CG_RM'],vmin=-vmax,vmax=vmax,s=60,cmap='RdBu_r',edgecolors= "k")

for i in range(0,4):
    axs[i].set_ylabel(' ',fontsize=fs)
    axs[i].set_xlabel(' ',fontsize=fs)

    axs[i].set_xlim(WCS(hdr).world_to_pixel(crange)[0])
    axs[i].set_ylim(WCS(hdr).world_to_pixel(crange)[1])

    #cbar = fig.colorbar(ims[i],ax=axs[i],orientation='vertical',fraction=0.012,pad=0.02,aspect=20)
    #cbar.ax.tick_params(axis='y', which='both',labelsize=fs)
    #cbar.set_label('[rad]', fontsize=fs)

    axs[i].tick_params(axis='both', which='major', labelsize=fs, zorder=30, length=5)

## Ratios of point source to diffuse versus longitude

In [ ]:
fix,axs = plt.subplots(2,2,figsize=(16,10))

axs[0,0].scatter(lon_sub,RM_sub/beams['G_FD'])
axs[0,1].scatter(lon_sub,RM_sub/beams['G_RM'])
axs[1,0].scatter(lon_sub,RM_sub/beams['C_RM'])
axs[1,1].scatter(lon_sub,RM_sub/beams['CG_RM'])

axs[0,0].set_ylabel(r'GMIMS peak FD')
axs[0,1].set_ylabel(r'GMIMS linear fit RM')
axs[1,0].set_ylabel(r'CGPS RM')
axs[1,1].set_ylabel(r'CGPS+GMIMS RM')

for i in range(0,2):
    for j in range(0,2):
        axs[j,i].set_xlim(82,52)
        axs[j,i].set_ylim(-20,20)
        axs[j,i].grid()
        axs[j,i].set_xlabel(r'Longitude (deg)')